In [18]:
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"  # useful if the file is on a shared filesystem
import h5py
import numpy as np
from scipy.io import loadmat
import os
import glob
import pandas as pd
from tqdm import tqdm
import re

In [7]:
def load_timestamp_group(base, mouse_num, stream_kind):
    """
    Load and concatenate copied timestamp CSVs for one mouse.

    stream_kind: 'BehavCam' or 'Miniscope'
    """
    folder = os.path.join(base, f"timeStamps{stream_kind}_copied_{mouse_num}")
    files = sorted(glob.glob(os.path.join(folder, "*.csv")))

    if not files:
        raise FileNotFoundError(f"No files found in {folder}")

    dfs = []
    suffix = f"_timeStamps{stream_kind}.csv"

    for file_i, path in enumerate(files):
        df = pd.read_csv(path).copy()
        source_file = os.path.basename(path)
        session_name = source_file.replace(suffix, "")

        df["source_file"] = source_file
        df["source_path"] = path
        df["session_name"] = session_name
        df["file_index"] = file_i
        df["frame_in_file"] = df["Frame Number"]
        dfs.append(df)

    out = pd.concat(dfs, ignore_index=True)
    out["cumulative_frame_index"] = np.arange(len(out))
    return out

In [8]:
def align_miniscope_to_behav_by_session(
    miniscope_df,
    behav_df,
    session_col="session_name",
    time_col="Time Stamp (ms)",
    frame_col="Frame Number",
    tolerance_ms=None,
    show_progress=True,
):
    mini = miniscope_df.copy()
    beh = behav_df.copy()

    mini[time_col] = pd.to_numeric(mini[time_col], errors="coerce")
    beh[time_col] = pd.to_numeric(beh[time_col], errors="coerce")

    mini["_orig_order"] = np.arange(len(mini))

    beh = beh.rename(
        columns={
            frame_col: "closestBehavCamFrameNumber",
            "frame_in_file": "closestBehavCamFrameIdx",
            "cumulative_frame_index": "closestBehavCamCumulativeIdx",
            time_col: "sys_clock_BehavCamFrame",
            "source_file": "behav_source_file",
            "source_path": "behav_source_path",
        }
    )

    beh_cols = [
        session_col,
        "sys_clock_BehavCamFrame",
        "closestBehavCamFrameNumber",
        "closestBehavCamFrameIdx",
        "closestBehavCamCumulativeIdx",
        "behav_source_file",
        "behav_source_path",
    ]

    sessions = mini[session_col].dropna().unique().tolist()
    iterator = tqdm(sessions, desc="Aligning sessions") if show_progress else sessions

    aligned_parts = []

    for session in iterator:
        mini_s = mini.loc[mini[session_col] == session].copy()
        beh_s = beh.loc[beh[session_col] == session, beh_cols].copy()

        if beh_s.empty:
            mini_s["closestBehavCamFrameNumber"] = np.nan
            mini_s["closestBehavCamFrameIdx"] = np.nan
            mini_s["closestBehavCamCumulativeIdx"] = np.nan
            mini_s["sys_clock_BehavCamFrame"] = np.nan
            mini_s["behav_source_file"] = np.nan
            mini_s["behav_source_path"] = np.nan
            mini_s["abs_time_diff_ms"] = np.nan
            aligned_parts.append(mini_s)
            continue

        mini_s = mini_s.sort_values(time_col)
        beh_s = beh_s.sort_values("sys_clock_BehavCamFrame")

        aligned_s = pd.merge_asof(
            mini_s,
            beh_s,
            left_on=time_col,
            right_on="sys_clock_BehavCamFrame",
            direction="nearest",
            tolerance=tolerance_ms,
        )

        aligned_s["abs_time_diff_ms"] = (
            aligned_s[time_col] - aligned_s["sys_clock_BehavCamFrame"]
        ).abs()

        aligned_parts.append(aligned_s)

    aligned = pd.concat(aligned_parts, ignore_index=True)
    aligned = aligned.sort_values("_orig_order").drop(columns="_orig_order").reset_index(drop=True)

    return aligned


In [23]:
def build_sourcefile_alignment_table(aligned_miniscope_df, aligned_csv_dir):
    cols = aligned_miniscope_df.columns.tolist()

    if "source_file" not in cols:
        raise KeyError(f"'source_file' not found. Available columns: {cols}")

    session_col = None
    for c in ["session_name", "session_name_x", "session_name_y"]:
        if c in cols:
            session_col = c
            break

    keep_cols = ["source_file"]
    if session_col is not None:
        keep_cols.append(session_col)

    source_sessions = (
        aligned_miniscope_df[keep_cols]
        .drop_duplicates()
        .copy()
    )

    if session_col is not None and session_col != "session_name":
        source_sessions = source_sessions.rename(columns={session_col: "session_name"})

    if "session_name" not in source_sessions.columns:
        source_sessions["session_name"] = source_sessions["source_file"].str.replace(
            "_timeStampsMiniscope.csv", "", regex=False
        )

    source_sessions["source_date"] = source_sessions["source_file"].str.extract(r"(\d{4}_\d{2}_\d{2})")
    source_sessions["date_key"] = source_sessions["source_date"].str.replace("_", "", regex=False)

    aligned_files = sorted(glob.glob(os.path.join(aligned_csv_dir, "*.csv")))
    aligned_df = pd.DataFrame({
        "aligned_csv_path": aligned_files,
        "aligned_csv_file": [os.path.basename(f) for f in aligned_files],
    })

    aligned_df["date_key"] = aligned_df["aligned_csv_file"].str.extract(r"(\d{8})")
    aligned_df["aligned_date"] = pd.to_datetime(aligned_df["date_key"], format="%Y%m%d", errors="coerce")

    match_table = source_sessions.merge(aligned_df, on="date_key", how="left")
    match_table["matched"] = match_table["aligned_csv_file"].notna()

    return match_table.sort_values(["source_date", "session_name"]).reset_index(drop=True)


In [9]:
# get indicies of "good" cells from ActSort 
labels_mat_path = r"/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/LinearTrackDataAlignedAcrossDays/m992_Yiwen/combinedDenoisedMatFiles_ds1/20260420_004618_068/precomputed_output_LABELS.mat"
mat = loadmat(labels_mat_path, struct_as_record=False, squeeze_me=True)
labels_struct = mat["labels"]
labels_ex = np.asarray(labels_struct.labels_ex).squeeze()
cell_indexes_to_keep = np.flatnonzero(labels_ex == 1)
print(cell_indexes_to_keep[:10])
print(len(cell_indexes_to_keep))
cell_indexes_to_keep_py = np.flatnonzero(labels_ex == 1)   # 0-based
cell_indexes_to_keep_matlab = cell_indexes_to_keep_py + 1  # 1-based

[ 2  3  4  5  6  7  8  9 10 11]
796


In [10]:
mat_path = r"/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/LinearTrackDataAlignedAcrossDays/m992_Yiwen/combinedDenoisedMatFiles_ds1/20260420_004618_068/day_013_2025_01_10_992_20_15_47_denoised_converted_concat_mc_Aligned_20260420_004618_068.mat"
with h5py.File(mat_path, "r") as f:
    temporal_weights = f["output"]["temporal_weights"][:]

print(temporal_weights.shape)
print(temporal_weights.dtype)

(943, 291847)
float32


In [11]:
actSortLabels = 'precomputed_output_LABELS.mat'

In [ ]:
## for alignment to ezTrack Data : 
# 1)find the msTime stamps files for each recording in this EXTRACT analysis 
# 2)create a cumulative list of frames, and then "extract" frames for a given day of recording 
base = "/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/YZ/Miniscope_data/Miniscope_data/Linear_track"
mouse_num = "992"
behav_df = load_timestamp_group(base, mouse_num, "BehavCam")
miniscope_df = load_timestamp_group(base, mouse_num, "Miniscope")

aligned_miniscope_df = align_miniscope_to_behav_by_session(
    miniscope_df,
    behav_df,
    tolerance_ms=None,   # or e.g. 100 if you want to reject bad matches
)

In [24]:
# 3) load velocity / position data, either from predone aligned .csvs or from ezTrack output
aligned_csv_dir = "/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/Linear track_aligned CSVs/1.75-m track/WT/#992_GCaMP8m"
match_table = build_sourcefile_alignment_table(aligned_miniscope_df, aligned_csv_dir)
match_table

,source_file,session_name,source_date,date_key,aligned_csv_path,aligned_csv_file,aligned_date,matched
0,2024_12_30_992_19_59_56_timeStampsMiniscope.csv,2024_12_30_992_19_59_56,2024_12_30,20241230,NaN,NaN,NaT,False
1,2024_12_30_992_20_24_07_timeStampsMiniscope.csv,2024_12_30_992_20_24_07,2024_12_30,20241230,NaN,NaN,NaT,False
2,2024_12_31_992_18_07_48_timeStampsMiniscope.csv,2024_12_31_992_18_07_48,2024_12_31,20241231,NaN,NaN,NaT,False
3,2024_12_31_992_18_28_21_timeStampsMiniscope.csv,2024_12_31_992_18_28_21,2024_12_31,20241231,NaN,NaN,NaT,False
4,2025_01_01_992_16_16_54_timeStampsMiniscope.csv,2025_01_01_992_16_16_54,2025_01_01,20250101,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,992_Run1_20250101_cellTracesAlignedToTracking.csv,2025-01-01,True
5,2025_01_02_992_18_58_39_timeStampsMiniscope.csv,2025_01_02_992_18_58_39,2025_01_02,20250102,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,992_Run1_20250102_cellTracesAlignedToTracking.csv,2025-01-02,True
6,2025_01_03_992_16_32_57_timeStampsMiniscope.csv,2025_01_03_992_16_32_57,2025_01_03,20250103,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,992_Run2_20250103_cellTracesAlignedToTracking.csv,2025-01-03,True
7,2025_01_04_992_18_03_51_timeStampsMiniscope.csv,2025_01_04_992_18_03_51,2025_01_04,20250104,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,992_Run3_20250104_cellTracesAlignedToTracking.csv,2025-01-04,True
8,2025_01_05_992_19_43_44_timeStampsMiniscope.csv,2025_01_05_992_19_43_44,2025_01_05,20250105,NaN,NaN,NaT,False
9,2025_01_05_992_19_58_29_timeStampsMiniscope.csv,2025_01_05_992_19_58_29,2025_01_05,20250105,NaN,NaN,NaT,False


In [17]:
## to query 1 day 
aligned_miniscope_df[aligned_miniscope_df["source_file"] == "2024_12_30_992_19_59_56_timeStampsMiniscope.csv"]


,Frame Number,Time Stamp (ms),Buffer Index,source_file,source_path,session_name_x,file_index,frame_in_file,cumulative_frame_index,session_name_y,sys_clock_BehavCamFrame,closestBehavCamFrameNumber,closestBehavCamFrameIdx,closestBehavCamCumulativeIdx,behav_source_file,behav_source_path,abs_time_diff_ms
0,0,-3,0,2024_12_30_992_19_59_56_timeStampsMiniscope.csv,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,2024_12_30_992_19_59_56,0,0,0,2024_12_30_992_19_59_56,22,1,1,1,2024_12_30_992_19_59_56_timeStampsBehavCam.csv,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,25
1,1,56,0,2024_12_30_992_19_59_56_timeStampsMiniscope.csv,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,2024_12_30_992_19_59_56,0,1,1,2024_12_30_992_19_59_56,89,2,2,2,2024_12_30_992_19_59_56_timeStampsBehavCam.csv,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,33
2,2,101,0,2024_12_30_992_19_59_56_timeStampsMiniscope.csv,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,2024_12_30_992_19_59_56,0,2,2,2024_12_30_992_19_59_56,89,2,2,2,2024_12_30_992_19_59_56_timeStampsBehavCam.csv,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,12
3,3,150,0,2024_12_30_992_19_59_56_timeStampsMiniscope.csv,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,2024_12_30_992_19_59_56,0,3,3,2024_12_30_992_19_59_56,167,3,3,3,2024_12_30_992_19_59_56_timeStampsBehavCam.csv,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,17
4,4,201,0,2024_12_30_992_19_59_56_timeStampsMiniscope.csv,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,2024_12_30_992_19_59_56,0,4,4,2024_12_30_992_19_59_56,230,4,4,4,2024_12_30_992_19_59_56_timeStampsBehavCam.csv,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21213,21213,1075474,0,2024_12_30_992_19_59_56_timeStampsMiniscope.csv,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,2024_12_30_992_19_59_56,0,21213,21213,2024_12_30_992_19_59_56,1075446,15418,15418,15418,2024_12_30_992_19_59_56_timeStampsBehavCam.csv,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,28
21214,21214,1075527,0,2024_12_30_992_19_59_56_timeStampsMiniscope.csv,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,2024_12_30_992_19_59_56,0,21214,21214,2024_12_30_992_19_59_56,1075528,15419,15419,15419,2024_12_30_992_19_59_56_timeStampsBehavCam.csv,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,1
21215,21215,1075575,0,2024_12_30_992_19_59_56_timeStampsMiniscope.csv,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,2024_12_30_992_19_59_56,0,21215,21215,2024_12_30_992_19_59_56,1075589,15420,15420,15420,2024_12_30_992_19_59_56_timeStampsBehavCam.csv,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,14
21216,21216,1075627,0,2024_12_30_992_19_59_56_timeStampsMiniscope.csv,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,2024_12_30_992_19_59_56,0,21216,21216,2024_12_30_992_19_59_56,1075589,15420,15420,15420,2024_12_30_992_19_59_56_timeStampsBehavCam.csv,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,38
